In [305]:
import logging
import time
import pandas as pd
import requests
import os

In [306]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("blue-owls-assessment") \
    .getOrCreate()

spark.version  # confirm Spark is running

'3.5.0'

In [307]:
API_BASE_URL = os.environ.get("API_BASE_URL", "http://mock-api:8000")
API_USERNAME = os.environ.get("API_USERNAME", "candidate")
API_PASSWORD = os.environ.get("API_PASSWORD", "blue-owls-2026")

In [308]:
class APIClient:
    def __init__(self):
        self.token = None
        self.max_retries = 6
        self.backoff = 1.5

    def authenticate(self):
        resp = requests.post(f"http://mock-api:8000/api/v1/auth/token", json={
            "username": API_USERNAME,
            "password": API_PASSWORD
        })

        resp.raise_for_status()
        self.token = resp.json()["access_token"]

    def get_headers(self):
        return {"Authorization": f"Bearer {self.token}"}

    def request(self, method, endpoint, params=None, data=None):
        # If no token exists, authenticate first
        if not self.token:
            self.authenticate()

        url = endpoint

        for attempt in range(self.max_retries+1):

            resp = requests.request(
            method=method,
            url=url,
            headers=self.get_headers(),
            params=params,
            json=data
        )

            # If token expired, refresh token automatically
            if resp.status_code == 401:
                print("⚠️ Token expired. Refreshing token...")
                self.authenticate()

                # Retry request with new token
                resp = requests.request(
                    method=method,
                    url=url,
                    headers=self.get_headers(),
                    params=params,
                    json=data
                )
            if resp.status_code == 429 or 500 <= resp.status_code < 600:
                    sleep_time = self.backoff ** attempt
                    print(f"Attempt {attempt} Status {resp.status_code}. Retrying in {sleep_time:.2f}s...")
                    time.sleep(sleep_time)
                    continue

            resp.raise_for_status()
            return resp

In [309]:
client = APIClient()

In [310]:


params = {"page_size":1000}
items_key= 'data'


endpoints = {"orders": '/api/v1/data/orders/?date_from=2018-07-01', "order_items": '/api/v1/data/order_items/?date_from=2018-07-01',
             "customers": '/api/v1/data/customers/?date_from=2018-07-01', "products": '/api/v1/data/products/?date_from=2018-07-01',
             "sellers": '/api/v1/data/sellers/?date_from=2018-07-01',
             "payments": '/api/v1/data/payments/?date_from=2018-07-01'}

def get_all_api_data(endpoint):
    """ Handles APIs using page numbers and returning total_pages or has_next flag"""
    page = 1
    all_items = []
    while True:
        params['page']= page
        resp = client.request("GET", f"{API_BASE_URL}{endpoint}", params=params)
        data = resp.json()
        page_items = data.get(items_key, [])
        if not page_items:
            break
        all_items.extend(page_items)
        total_pages = data.get("total_pages")
        if total_pages:
            if page >= total_pages:
                break
        if data.get("has_next") is False:
            break
        page += 1
    return all_items

In [312]:
def save_to_csv(items, csv_path="output.csv", source_endpoint=""):
    if not items:
        print("No items fetched")
    df = spark.createDataFrame(items)
    df = df.dropDuplicates(df.columns)
    try:
        existing_data = spark.read.csv(csv_path, header=True, inferSchema=True)
        ex_cols = existing_data.remove("_ingested_at")
        ex_cols = existing_data.remove("_source_endpoint")
        existing_data = existing_data.dropDuplicates(existing_data.columns)
    except Exception:
        existing_data = spark.createDataFrame([], schema=df.schema)
    cols = df.columns
    df_to_append = df.join(existing_data, on=cols, how="left_anti")
    df_to_append = df_to_append.withColumn("_ingested_at", current_timestamp()).withColumn("_source_endpoint", lit(source_endpoint))
    df_to_append = df_to_append.distinct()
    df_to_append.coalesce(1).write.mode("append").csv(csv_path, header=True)
    print(f"appended to {csv_path} with records {df_to_append.count()}")

In [313]:
for i in endpoints.keys():
    items = get_all_api_data(endpoints.get(i))
    save_to_csv(items, csv_path=f"work/output/bronze/{i}.csv", source_endpoint=i)

Attempt 0 Status 429. Retrying in 1.00s...
Attempt 1 Status 429. Retrying in 1.50s...
Attempt 2 Status 500. Retrying in 2.25s...
Attempt 3 Status 500. Retrying in 3.38s...
Attempt 0 Status 429. Retrying in 1.00s...
Attempt 1 Status 429. Retrying in 1.50s...
Attempt 2 Status 500. Retrying in 2.25s...
Attempt 0 Status 500. Retrying in 1.00s...
Attempt 1 Status 500. Retrying in 1.50s...
Attempt 2 Status 429. Retrying in 2.25s...
Attempt 0 Status 500. Retrying in 1.00s...
Attempt 0 Status 429. Retrying in 1.00s...
Attempt 1 Status 429. Retrying in 1.50s...
Attempt 0 Status 500. Retrying in 1.00s...
Attempt 0 Status 500. Retrying in 1.00s...
Attempt 1 Status 429. Retrying in 1.50s...
Attempt 2 Status 429. Retrying in 2.25s...
appended to work/output/bronze/orders.csv with records 12824
Attempt 0 Status 429. Retrying in 1.00s...
Attempt 1 Status 429. Retrying in 1.50s...
Attempt 0 Status 500. Retrying in 1.00s...
Attempt 1 Status 429. Retrying in 1.50s...
Attempt 2 Status 429. Retrying in 2.

Attempt 0 Status 429. Retrying in 1.00s...
Attempt 1 Status 429. Retrying in 1.50s...
Attempt 2 Status 500. Retrying in 2.25s...
Attempt 0 Status 500. Retrying in 1.00s...
Attempt 1 Status 500. Retrying in 1.50s...
Attempt 2 Status 429. Retrying in 2.25s...
Attempt 0 Status 429. Retrying in 1.00s...
Attempt 1 Status 429. Retrying in 1.50s...
Attempt 2 Status 500. Retrying in 2.25s...
Attempt 0 Status 500. Retrying in 1.00s...
Attempt 1 Status 429. Retrying in 1.50s...
Attempt 2 Status 429. Retrying in 2.25s...
Attempt 0 Status 429. Retrying in 1.00s...
Attempt 1 Status 429. Retrying in 1.50s...
Attempt 0 Status 500. Retrying in 1.00s...
Attempt 1 Status 429. Retrying in 1.50s...
Attempt 2 Status 429. Retrying in 2.25s...
Attempt 0 Status 500. Retrying in 1.00s...
Attempt 0 Status 500. Retrying in 1.00s...
Attempt 1 Status 429. Retrying in 1.50s...
Attempt 2 Status 429. Retrying in 2.25s...
Attempt 0 Status 500. Retrying in 1.00s...
Attempt 1 Status 500. Retrying in 1.50s...
Attempt 0 S